# Lab 19: DDP and MLflow

**Module 19** | Read `notes/19-ddp-mlops.pdf` before running this notebook.

Train a small CNN on MNIST while logging parameters, metrics, and the model artifact to MLflow.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Experiment Tracking with MLflow

MLOps workflows need reproducible records of **what was trained, with which hyperparameters, and how well it performed**.
[MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) logs params, metrics, and artifacts (including PyTorch models) to a local or remote store.

This lab trains a **small CNN on MNIST** for a few epochs and logs hyperparameters, per-epoch loss, test accuracy, and the model artifact.

The tracking URI comes from the environment (`MLFLOW_TRACKING_URI`), set by `runtime_env.configure_runtime()` to a local SQLite store under `mlruns/mlflow.db`.
Browse runs at http://127.0.0.1:5050 after launching the MLflow UI via `env.ps1` or `env.sh`.


### Model architecture

A compact CNN with two convolutional blocks, pooling, and a fully connected head. The same class is used for training and for the MLflow model artifact.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import os

import torch
import torch.nn as nn


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


model_preview = SmallCNN()
param_count = sum(p.numel() for p in model_preview.parameters())
print(f"SmallCNN parameter count: {param_count:,}")
print(model_preview)


### Load MNIST and build DataLoaders

We download MNIST into `datasets/mnist`, normalize pixels, and inspect the shape of the first training batch.


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

data_root = ROOT / "datasets" / "mnist"
train_ds = datasets.MNIST(str(data_root), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(str(data_root), train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f"Training device: {device}")
print(f"Train samples: {len(train_ds):,}, test samples: {len(test_ds):,}")
print(f"Batch image shape: {tuple(xb.shape)}  (batch, channels, height, width)")
print(f"Batch label shape: {tuple(yb.shape)}")
print(f"Sample batch labels (first 16): {yb[:16].tolist()}")
print(f"Label dtype: {yb.dtype}, value range: {int(yb.min())} to {int(yb.max())}")

epochs = 2
lr = 1e-3


### Train, evaluate, and log to MLflow

Each epoch prints average training loss and training accuracy.
After training we compute test accuracy, log everything to MLflow, and inspect 10 sample predictions with digit probabilities.


In [ ]:
import mlflow
import mlflow.pytorch

tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

with mlflow.start_run(run_name="lab19_mnist_cnn") as run:
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("learning_rate", lr)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("device", str(device))
    mlflow.log_param("architecture", "SmallCNN")

    model = SmallCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print("=" * 50)
    print("TRAINING")
    print("=" * 50)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("train_accuracy", epoch_acc, step=epoch)
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"loss={epoch_loss:.4f}, train accuracy={epoch_acc:.4f}"
        )

    model.eval()
    test_correct = 0
    test_total = 0
    all_logits: list[torch.Tensor] = []
    all_labels: list[torch.Tensor] = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            test_correct += (preds == yb).sum().item()
            test_total += yb.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(yb.cpu())

    test_acc = test_correct / test_total
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.pytorch.log_model(model, artifact_path="mnist_cnn")

    print()
    print("=" * 50)
    print("TEST SET")
    print("=" * 50)
    print(f"Test accuracy: {test_acc:.4f} ({test_correct}/{test_total} correct)")

    logits_cat = torch.cat(all_logits, dim=0)
    labels_cat = torch.cat(all_labels, dim=0)
    probs = torch.softmax(logits_cat[:10], dim=1)

    print()
    print("Sample predictions (first 10 test images):")
    print(f"{'Idx':>4} | {'True':>4} | {'Pred':>4} | Top-3 probabilities (digit: prob)")
    print("-" * 60)
    for i in range(10):
        true_digit = int(labels_cat[i])
        pred_digit = int(logits_cat[i].argmax())
        top3 = probs[i].topk(3)
        top3_str = ", ".join(
            f"{int(d)}:{p:.3f}" for d, p in zip(top3.indices, top3.values)
        )
        print(f"{i:>4} | {true_digit:>4} | {pred_digit:>4} | {top3_str}")

    print()
    print("=" * 50)
    print("MLFLOW RUN INFO")
    print("=" * 50)
    print(f"Run ID: {run.info.run_id}")
    print(f"Run name: {run.info.run_name}")
    print(f"Experiment ID: {run.info.experiment_id}")
    print(f"Artifact URI: {mlflow.get_artifact_uri('mnist_cnn')}")
    mlflow_port = os.environ.get("MLFLOW_UI_PORT", "5050")
    print(f"Open MLflow UI at http://127.0.0.1:{mlflow_port} to inspect params, metrics, and model.")


## Optional: Distributed Data Parallel (DDP)

For multi-GPU training, PyTorch **DistributedDataParallel** wraps the model and synchronizes gradients across processes launched with `torchrun`:

```bash
torchrun --nproc_per_node=2 train_script.py
```

Each process runs on one GPU; only rank 0 should call `mlflow.log_*` to avoid duplicate runs.
This lab uses single-process training so it runs on any machine; the MLflow logging pattern is the same in DDP, only the launch command differs.

**Deliverable:** note the printed **run ID** and confirm the run appears in the MLflow UI with `test_accuracy` and a `mnist_cnn` model artifact.
